# imgjit — Colab build/test runner

Runtime must be set to **T4 GPU** (`Runtime → Change runtime type`) before running this notebook.

See `docs/PLAN.md` for which phase this is verifying and `docs/ARCHITECTURE.md` / `docs/PROTOCOL.md`
for design context. Run cells top to bottom; the clone/pull cell is safe to re-run after every
`git push` from your Mac — no need to restart the runtime.

In [ ]:
import os

REPO_URL = "https://github.com/zmx27/Image-Processing-Engine.git"
REPO_DIR = "/content/Image-Processing-Engine"

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

## Toolchain / GPU sanity check

Run once per session to confirm you actually got a GPU and to catch CUDA/driver version drift
between sessions before it shows up as a confusing build or runtime failure.

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version

## Configure + build

`find_package(CUDAToolkit)` in the root `CMakeLists.txt` should report the CUDA backend enabled
here (unlike on a Mac, which always falls back to the CPU-only portable core).

In [ ]:
%cd {REPO_DIR}
!cmake -B build -DCMAKE_BUILD_TYPE=RelWithDebInfo
!cmake --build build -j

## Run tests

In [ ]:
%cd {REPO_DIR}
!ctest --test-dir build --output-on-failure

## Phase-specific runs

Cells below are added incrementally as each phase in `docs/PLAN.md` produces something runnable
(e.g. the Phase 2/3 `imgjit-cli`, the Phase 8 `bench` harness).

### Phase 1 — driver API + JIT spike

`imgjit-spike` inverts a PNG on-GPU twice — once from PTX that `nvcc` built ahead of time
(checkpoint 1a, driver plumbing with JIT out of the picture), then once from PTX that NVRTC
compiled at runtime (checkpoint 1b) — and diffs both against a scalar CPU loop. Inversion is an
integer pointwise op, so the bar is exact equality. It exits non-zero on any mismatch, so the
`ctest` cell above already covers it; run it directly to see the per-checkpoint output and to
inspect the generated PTX.

In [ ]:
%cd {REPO_DIR}
!./build/tools/imgjit-spike tests/testdata/gradient_32x32_rgba.png \
    --out /content/inverted.png --dump-ptx /content/invert_jit.ptx

# The generated PTX is what you read when debugging codegen (Phase 3 adds --dump-source
# for the CUDA side of the same idea).
!head -30 /content/invert_jit.ptx

### Phase 3 — codegen + kernel cache

`imgjit-cli --backend cuda` runs the chain through NVRTC codegen, the kernel cache and the driver
API. `--dump-source` writes the *generated* kernel, which is what you read when debugging — not the
fragments it was assembled from — and `--repeat` makes the cold-compile / warm-hit split visible:
run 1 pays the ~50-200 ms NVRTC compile, runs 2..n hit the cache.

Correctness against the CPU oracle and the compile-counter assertions are `imgjit-gpu-tests`,
already run by the `ctest` cell above as `phase3_gpu_oracle_diff` and `phase3_gpu_kernel_cache`.


In [ ]:
%cd {REPO_DIR}
!./build/tools/imgjit-cli --backend cuda \
    --ops "grayscale,gaussian:1.4,sobel,threshold:0.3" --repeat 5 \
    --dump-source /content/chain.cu --dump-ptx /content/chain.ptx \
    tests/testdata/gradient_32x32_rgba.png /content/phase3_out.png

# The generated kernel. Two stages here: grayscale folds into the gaussian's tap helper
# (a prologue, recomputed per tap), and the threshold folds into the sobel's epilogue.
!cat /content/chain.cu


### Phase 5 — GPU worker behind the server

`imgjit-server --backend cuda` puts the CUDA backend on the worker thread: `cuCtxCreate` once at
startup on that thread, the frame slots allocated as **pinned** host memory that connection threads
`recv()` straight into, and every copy and launch on a **single stream** — deliberately, so a
failure here is unambiguously plumbing rather than async overlap (that is Phase 6).

The correctness gate is `phase5_gpu_server`, already run by the `ctest` cell above.

The two cells below record the Phase 5 baseline in `bench/baseline_phase5.csv`. Only re-run
these if you mean to refresh it — and `rm bench/baseline_phase5.csv` first if so. The Phase 6
comparison does **not** depend on re-running these; it writes its own file (`baseline_phase6.csv`)
with a `--streams 1` row that is the honest current-code equivalent of the Phase 5 pipeline.


In [ ]:
%cd {REPO_DIR}
import subprocess, time, re, os

def start_server(extra_args):
    log = open('/content/server.log', 'w+')
    proc = subprocess.Popen(
        ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
         '--slots', '8', '--slots-per-conn', '2', '--queue', '16',
         '--max-payload', str(16 * 1024 * 1024), '--output-dir', '/content/phase5_out'] + extra_args,
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(200):
        log.flush()
        text = open('/content/server.log').read()
        match = re.search(r'127\.0\.0\.1:(\d+)', text)
        if match:
            print(text.strip())
            return proc, int(match.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())

def bench(port, label):
    subprocess.run(
        ['./build/bench/imgjit-bench', '--port', str(port), '--connections', '4',
         '--frames', '200', '--width', '1024', '--height', '1024', '--channels', '3',
         '--ops', 'grayscale,gaussian:1.4,sobel,threshold:0.3',
         '--label', label, '--csv', 'bench/baseline_phase5.csv'], check=True)

# Run 1: cold. The first frame on each connection pays the NVRTC compile.
proc, port = start_server([])
bench(port, 'phase5_cuda_cold')
proc.terminate(); proc.wait()
print(open('/content/server.log').read().strip().splitlines()[-1])

In [ ]:
%cd {REPO_DIR}
# Run 2: the same load against a server that compiled the chain at startup. Only
# `cold_first_ms` should move — which is docs/ARCHITECTURE.md's prewarm claim, measured.
proc, port = start_server(['--prewarm', 'grayscale,gaussian:1.4,sobel,threshold:0.3'])
bench(port, 'phase5_cuda_prewarmed')
proc.terminate(); proc.wait()
print(open('/content/server.log').read().strip().splitlines()[-1])

print()
print(open('bench/baseline_phase5.csv').read())

### Phase 6 — async multi-stream pipeline

`--streams` is the A/B axis. `--streams 1` is the Phase 5 pipeline (one frame on the GPU at a
time); `--streams 4` is the Phase 6 one. Everything else about the two runs is identical, so the
comparison is like-for-like under the same harness rather than against a build that no longer
exists.

Read **both** outputs. The CSV answers "measurably faster"; the server's last line answers the
other half of the gate — mean and peak stream occupancy. A `mean in flight` near 1.0 under
`--streams 4` means the pipeline serialized and the throughput came from somewhere else, which is
a failed gate no matter what the FPS says.

The correctness gates are `phase6_gpu_async`, `phase6_gpu_stress` and `phase6_gpu_recovery`,
already run by the `ctest` cell above.


In [ ]:
%cd {REPO_DIR}
# Self-contained: does not depend on the Phase 5 cells having run.
import subprocess, time, re, os

# The full showcase chain. Compute-heavy (gaussian:1.4 is ~11x11 taps/pixel), so the
# copy time overlap can hide is a modest fraction of each frame — expect ~1.2-1.4x, not
# 2x. The cheap chain below makes the copy-bound case, where overlap wins big.
CHAIN = 'grayscale,gaussian:1.4,sobel,threshold:0.3'
CHEAP = 'invert'

def start_server(streams):
    log = open('/content/server.log', 'w+')
    proc = subprocess.Popen(
        ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
         '--streams', str(streams),
         # Wide enough that the GPU, not the slot pool, is the bottleneck: 4 conns x
         # window 8 = 32 frames offered, and 32 slots / 8-per-conn lets them all land.
         '--slots', '32', '--slots-per-conn', '8', '--queue', '32',
         '--max-payload', str(16 * 1024 * 1024), '--output-dir', '/content/phase6_out',
         '--prewarm', CHAIN, '--prewarm', CHEAP],
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        log.flush()
        m = re.search(r'127\.0\.0\.1:(\d+)', open('/content/server.log').read())
        if m:
            print(open('/content/server.log').read().strip())
            return proc, int(m.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())

def bench6(port, label, chain):
    subprocess.run(
        ['./build/bench/imgjit-bench', '--port', str(port), '--connections', '4', '--window', '8',
         '--frames', '300', '--width', '1024', '--height', '1024', '--channels', '3',
         '--ops', chain, '--label', label, '--csv', 'bench/baseline_phase6.csv'], check=True)

# imgjit-bench APPENDS a row per run — start clean so re-running does not duplicate rows.
!rm -f bench/baseline_phase6.csv

# streams 1 = the Phase 5 pipeline (one frame on the GPU at a time); streams 4 = Phase 6.
# Same load both times, so the fps gap is the effect of overlap. Read the server's last
# line too: mean-in-flight near 1.0 under streams 4 means the pipeline serialized.
for streams in (1, 4):
    for chain, tag in ((CHAIN, 'full'), (CHEAP, 'cheap')):
        proc, port = start_server(streams)
        bench6(port, f'phase6_streams{streams}_{tag}', chain)
        proc.terminate(); proc.wait()
        print('  '.join(open('/content/server.log').read().strip().splitlines()[-1:]))
        print()

print(open('bench/baseline_phase6.csv').read())


#### Nsight Systems — the overlap gate

Throughput alone cannot distinguish a pipeline that overlaps from one that got faster for another
reason, so the phase gate also asks for a timeline. The H2D, kernel and D2H rows must be genuinely
interleaved across the four streams, not a single file of segments separated by gaps.


In [ ]:
%cd {REPO_DIR}
# Self-contained. nsys is not always on PATH on Colab — it may only exist bundled inside
# nsight-compute. --trace=cuda only: CPU sampling needs perf permissions Colab withholds.
import subprocess, time, os, glob, shutil, signal

CHAIN = 'grayscale,gaussian:1.4,sobel,threshold:0.3'

_candidates = [shutil.which('nsys')]
for pat in ('/opt/nvidia/nsight-systems/*/target-linux-x64/nsys',
            '/opt/nvidia/nsight-compute/*/host/target-linux-x64/nsys',
            '/opt/nvidia/nsight-compute/*/target-linux-x64/nsys',
            '/usr/local/cuda*/bin/nsys'):
    _candidates += sorted(glob.glob(pat))
NSYS = next((c for c in _candidates if c and os.path.exists(c)), None)
assert NSYS, ('no nsys anywhere. The ctest and occupancy gates stand without it; '
              'produce the timeline later on any box with Nsight Systems.')
print('using', NSYS)
subprocess.run([NSYS, '--version'])

nsys_cmd = [
    NSYS, 'profile', '-o', '/content/phase6', '--trace=cuda', '--force-overwrite', 'true',
    './build/tools/imgjit-server', '--port', '9100', '--backend', 'cuda', '--streams', '4',
    '--slots', '32', '--slots-per-conn', '8', '--queue', '32',
    '--max-payload', str(16 * 1024 * 1024), '--prewarm', CHAIN,
]
log = open('/content/nsys_server.log', 'w+')
proc = subprocess.Popen(nsys_cmd, stdout=log, stderr=subprocess.STDOUT)
time.sleep(20)  # context creation + prewarm compile, before any frame arrives

subprocess.run(
    ['./build/bench/imgjit-bench', '--port', '9100', '--connections', '4', '--window', '8',
     '--frames', '300', '--width', '1024', '--height', '1024', '--channels', '3', '--ops', CHAIN,
     '--label', 'nsys_run'], check=True)

proc.send_signal(signal.SIGINT)   # clean shutdown, so nsys writes a complete report
proc.wait()
print(open('/content/nsys_server.log').read()[-2000:])

# Textual summary — paste into the commit message / PLAN. The real gate is visual:
# download /content/phase6.nsys-rep, open it in the Nsight Systems desktop app, and check
# the HtoD / kernel / DtoH rows interleave across the four streams rather than run single
# file. `nsys stats` here is the numeric shadow of that.
subprocess.run([NSYS, 'stats', '--report', 'cuda_gpu_trace', '/content/phase6.nsys-rep'])
